# Data Preprocessing — Market Sentiment Tweets

**Objetivo:** Limpar e normalizar os tweets do corpus de treino para preparar os dados para o Feature Engineering e Modelação.

Técnicas aplicadas (≥4 obrigatórias pelo enunciado):
1. Lowercasing
2. Remoção de URLs e @mentions
3. Tratamento de hashtags (mantém a palavra)
4. Normalização de cashtags ($TICKER)
5. Remoção de stopwords
6. Lemmatization
7. Stemming (alternativa à lemmatization)
8. Regex (limpeza de caracteres especiais)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk.stem.wordnet import WordNetLemmatizer

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',   quiet=True)
nltk.download('stopwords', quiet=True)

stop    = set(stopwords.words('english'))
lemma   = WordNetLemmatizer()
stemmer = SnowballStemmer('english')

print('✅ Imports OK')

In [ ]:
# ── Carregar dados ────────────────────────────────────────────────────────────
df = pd.read_csv('train.csv')

print(f'Shape: {df.shape}')
print(f'Colunas: {list(df.columns)}')
df.head(5)

ver qual fazemos texto = "I can't believe $TSLA isn't rising! Check www.tesla.com"

# .split() — divide apenas por espaços
texto.split()
# ['I', "can't", 'believe', '$TSLA', "isn't", 'rising!', 'Check', 'www.tesla.com']

# word_tokenize — usa regras linguísticas
word_tokenize(texto)
# ['I', 'ca', "n't", 'believe', '$', 'TSLA', 'is', "n't", 'rising', '!', 'Check', 'www.tesla.com']

## 1. Função de Limpeza — igual à das aulas, adaptada para tweets financeiros

Diferenças em relação ao Notebook 1 (Amazon Reviews):
- `#hashtag` → mantém a palavra (remove só o `#`)
- `@mention` → remove completamente
- `$TSLA` → substitui por `TICKER` (mantém info sem overfitting num ticker específico)
- As substituições especiais são feitas **antes** de remover caracteres não-alfabéticos

In [ ]:
# ── Função clean() ────────────────────────────────────────────────────────────
def clean(text_list, lemmatize=True, use_stemmer=False):
    """
    Limpa uma lista de tweets:
      1. Lowercase
      2. Remove URLs
      3. Remove @mentions
      4. Mantém palavras de #hashtags (remove só o #)
      5. Substitui cashtags ($TSLA) por TICKER
      6. Remove caracteres não-alfabéticos (regex)
      7. Remove stopwords
      8. Lemmatization ou Stemming
    """
    updates = []
    for text in tqdm(text_list, desc='Cleaning'):
        text = str(text)
        # ver se deixo estar isto , ou se as letras maiusculas têm algum significado (ex: "I LOVE THIS" vs "I love this") de sentiment0 
        # 1. Lowercase
        text = text.lower() # entao vou ver em cada label o nuemro de palavra ou letras grandes em cada label se for uma diferneca signifcativa 
        
        # se houver uma diferenca significativa entre as labels, ou seja se os tweets bullish tiverem muito mais palavras em maiusculo do que os bearish, então pode ser interessante manter essa informação. Por exemplo, se "I LOVE THIS" aparecer muito em bullish e "I hate this" aparecer muito em bearish, isso pode ser um sinal forte de sentimento. Por outro lado, se não houver uma diferença clara, ou se as palavras em maiusculo não forem um indicador confiável de sentimento, então pode ser melhor normalizar tudo para lowercase para reduzir o ruído e o vocabulário.
        
        # 2. Remove URLs
        text = re.sub(r'http\S+|www\S+', '', text)
        
        #ver se removo isto ou ver se identificar nomes grandes tipo elon musk esta associado a aglum de sentimento positivo ou negativo, exemplo elen mussk
        # VER NO DATA EXPLORATION QUAIS OS @ MAIS MENCIANDOS EM CADA LABEL SE HOUVER UMA DIFERENÇA SIGNIFICATIVA ENTRE AS LABELS, OU SE OS @MENTIONS FOREM UM INDICADOR CONFIÁVEL DE SENTIMENTO, ENTÃO PODE SER INTERESSANTE MANTER ESSA INFORMAÇÃO. POR EXEMPLO, SE "@ELONMUSK" APARECER MUITO EM BULLISH E "@SOMECRITIC" APARECER MUITO EM BEARISH, ISSO PODE SER UM SINAL FORTE DE SENTIMENTO. POR OUTRO LADO, SE NÃO HOUVER UMA DIFERENÇA CLARA, OU SE OS @MENTIONS NÃO FOREM UM INDICADOR CONFIÁVEL DE SENTIMENTO, ENTÃO PODE SER MELHOR REMOVER ESSA INFORMAÇÃO PARA REDUZIR O RUÍDO E O VOCABULÁRIO.
        # 3. Remove @mentions
        text = re.sub(r'@\w+', '', text)
        
        
        # FAZER COM HASTAGS A MESMA COISA COM OS ARROBAS 
        # 4. Hashtags — mantém a palavra, remove só o #
        text = re.sub(r'#(\w+)', r'\1', text)

        """

         PORQUE ISTO O  QUE FAZ É TRANSFORMA TUDO DE $TSLA, $AAPL, $MSFT, $AMZN, $GOOG, $META, $NVDA, $NFLX, $INTC, $AMD PARA TICKER, O QUE É BOM PORQUE O MODELO VAI APRENDER QUE HÁ UM TICKER CITADO, MAS NÃO VAI FICAR VICIADO EM APRENDER QUE $TSLA É SEMPRE POSITIVO OU NEGATIVO
        Os objetivos principais são:

Evitar overfitting a empresas específicas
Se o modelo aprender que $TSLA aparece muito em tweets positivos ou negativos, ele pode ficar demasiado dependente desse ticker em vez de aprender o padrão de sentimento.

Generalizar melhor
Ao substituir todos os cashtags pelo mesmo marcador, o modelo aprende que “há um ticker citado” em vez de memorizar “este ticker específico”.

Reduzir o vocabulário
Em vez de ter centenas ou milhares de tokens diferentes para ações diferentes, ficas com um token único. Isso simplifica o espaço de features.

Manter informação útil
O ticker em si pode ser relevante para sentimento financeiro, por isso removê-lo totalmente pode perder sinal. Substituir por ticker preserva o facto de existir uma menção financeira sem fixar a identidade da ação.
  
MAS VER SE NAO ESTOU A ROUBAR INFORMACAO , QUE SE TIVESSE A POR EMPRESA FICA ASSOCIADO A UM SENTIMENTO POR EXEMPLO 
TESLA SUPOSTAMENTE DEVIA FICAR ASSOCIADO A UM SNETIMENTO BOM MAS O PROBLEMA É QUE TAMBEM DE DEPENDE MUITO DA ALTURA TEMPORAL 
PORQUE DEPOIS A EMPRESA PODE ESTAR BOA NAQUELE MOMENTO E DEPOIS FICAR MA, O QUE TORNA DEPOIS COMPLEXO 
        """

        # 5. Cashtags — substitui $TSLA por TICKER
        text = re.sub(r'\$[a-zA-Z]{1,5}', 'ticker', text)
        #ve se certo caracteres nao exprimem sentimento Se o resultado mostrar que Bullish tem significativamente mais ! que Bearish, então vale a pena guardar essa informação antes de a remover:
        # 6. Remove caracteres não-alfabéticos (regex)
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)

        # 7. Remove stopwords
        tokens = word_tokenize(text)
        tokens = [w for w in tokens if w not in stop and len(w) > 2]
#usar  lematize 
        # 8a. Lemmatization
        if lemmatize:
            tokens = [lemma.lemmatize(w) for w in tokens]
        #se calhar so usamos uma das duas, ou seja ou lemmatization ou stemming, porque as duas fazem coisas parecidas, mas o stemming é mais agressivo e pode cortar palavras de forma que perca o significado, enquanto a lemmatization é mais cuidadosa e mantém a forma base da palavra. Se usarmos as duas, podemos acabar com palavras que são tão curtas que não têm significado (ex: "run" vira "run" com lemmatization, mas com stemming pode virar "ru"). Por isso, talvez seja melhor escolher uma abordagem e usá-la consistentemente.
        # 8b. Stemming (alternativa)
        if use_stemmer:
            tokens = [stemmer.stem(w) for w in tokens]
            #retirar os emojis 

        updates.append(' '.join(tokens))
    return updates

## 2. Demonstração passo a passo (como nas aulas)

In [ ]:
# ── Trace de um tweet exemplo ─────────────────────────────────────────────────
tweet_exemplo = "$TSLA beats earnings estimates! #Bullish #stocks @elonmusk https://t.co/abc123 up 5.3%"

t = tweet_exemplo
steps = [
    ('1. Original',              t),
    ('2. Lowercase',             t := t.lower()),
    ('3. Remove URLs',           t := re.sub(r'http\S+|www\S+', '', t)),
    ('4. Remove @mentions',      t := re.sub(r'@\w+', '', t)),
    ('5. Hashtags → palavras',   t := re.sub(r'#(\w+)', r'\1', t)),
    ('6. Cashtags → TICKER',     t := re.sub(r'\$[a-zA-Z]{1,5}', 'ticker', t)),
    ('7. Só letras (regex)',      t := re.sub(r'[^a-zA-Z\s]', ' ', t)),
    ('8. Remove stopwords',      t := ' '.join(w for w in t.split() if w not in stop and len(w) > 2)),
    ('9. Lemmatization',         t := ' '.join(lemma.lemmatize(w) for w in t.split())),
]

print('=' * 70)
for step_name, step_text in steps:
    print(f'{step_name:<30} → {step_text.strip()}')
print('=' * 70)

## 3. Comparação: Lemmatization vs Stemming (como nas aulas)

In [ ]:
# ── Lemmatization vs Stemming ─────────────────────────────────────────────────
# Palavras relevantes para o contexto financeiro
financial_words = [
    'beats', 'beating', 'beaten',
    'misses', 'missing', 'missed',
    'earnings', 'earning',
    'rises', 'rising', 'raised',
    'falling', 'fallen', 'falls',
    'bullish', 'bearish',
    'investing', 'investors',
]

print(f"{'Palavra':<18} {'Lemma':<18} {'Stem':<18}")
print('-' * 54)
for word in financial_words:
    l = lemma.lemmatize(word)
    s = stemmer.stem(word)
    diff = '  ← diferente!' if l != s else ''
    print(f'{word:<18} {l:<18} {s:<18}{diff}')

## 4. Aplicar o Preprocessing ao Dataset

In [ ]:
# ── Versão com Lemmatization (principal) ──────────────────────────────────────
print('A limpar com Lemmatization...')
df['text_lemma'] = clean(df['text'], lemmatize=True, use_stemmer=False)

# ── Versão com Stemming (comparação) ─────────────────────────────────────────
print('A limpar com Stemming...')
df['text_stem'] = clean(df['text'], lemmatize=False, use_stemmer=True)

print('\n✅ Preprocessing concluído!')
print(f'Colunas disponíveis: {list(df.columns)}')

In [ ]:
# ── Comparação antes/depois ───────────────────────────────────────────────────
print('Exemplos de tweets antes e depois do preprocessing:\n')
for i in range(5):
    print(f'[{i}] ORIGINAL : {df["text"].iloc[i]}')
    print(f'    LEMMA    : {df["text_lemma"].iloc[i]}')
    print(f'    STEM     : {df["text_stem"].iloc[i]}')
    print()

## 5. Verificar Tweets Vazios após Limpeza

In [ ]:
# ── Tweets vazios após limpeza ────────────────────────────────────────────────
vazios_lemma = (df['text_lemma'].str.strip() == '').sum()
vazios_stem  = (df['text_stem'].str.strip() == '').sum()
muito_curtos = (df['text_lemma'].str.split().str.len() < 3).sum()

print(f'Tweets vazios após limpeza (lemma): {vazios_lemma}')
print(f'Tweets vazios após limpeza (stem):  {vazios_stem}')
print(f'Tweets com menos de 3 palavras:     {muito_curtos}')

if vazios_lemma > 0:
    print('\nTweets originais que ficaram vazios:')
    mask = df['text_lemma'].str.strip() == ''
    print(df[mask][['text', 'label']].to_string())
    
    # Remover tweets vazios
    df = df[df['text_lemma'].str.strip() != ''].reset_index(drop=True)
    print(f'\nDataset após remover vazios: {len(df)} tweets')

## 6. Verificar Duplicados após Limpeza

In [ ]:
# ── Duplicados ────────────────────────────────────────────────────────────────
dup_original = df.duplicated(subset='text').sum()
dup_limpo    = df.duplicated(subset='text_lemma').sum()

print(f'Duplicados no texto original: {dup_original}')
print(f'Duplicados após limpeza:      {dup_limpo}')

# Remover duplicados (manter o primeiro)
if dup_original > 0:
    df = df.drop_duplicates(subset='text', keep='first').reset_index(drop=True)
    print(f'Dataset após remover duplicados: {len(df)} tweets')

## 7. Impacto no Vocabulário — antes vs depois

In [ ]:
# ── Comparação de vocabulário ─────────────────────────────────────────────────
from collections import Counter

def get_vocab(series):
    tokens = []
    for text in series:
        tokens.extend(str(text).split())
    return set(tokens), len(tokens)

vocab_orig,  total_orig  = get_vocab(df['text'])
vocab_lemma, total_lemma = get_vocab(df['text_lemma'])
vocab_stem,  total_stem  = get_vocab(df['text_stem'])

print(f"{'Versão':<20} {'Tokens totais':>15} {'Vocabulário único':>18}")
print('-' * 55)
print(f"{'Original':<20} {total_orig:>15,} {len(vocab_orig):>18,}")
print(f"{'Lemmatization':<20} {total_lemma:>15,} {len(vocab_lemma):>18,}")
print(f"{'Stemming':<20} {total_stem:>15,} {len(vocab_stem):>18,}")
print()
print(f'Redução de vocabulário (lemma vs original): {(1 - len(vocab_lemma)/len(vocab_orig))*100:.1f}%')
print(f'Redução de vocabulário (stem  vs original): {(1 - len(vocab_stem)/len(vocab_orig))*100:.1f}%')

## 8. Corpus Split — Stratified K-Fold

Dividimos o `train.csv` em **treino** e **validação** usando Stratified K-Fold para garantir que a proporção das classes se mantém em cada fold.

In [ ]:
# ── Stratified K-Fold ─────────────────────────────────────────────────────────
from sklearn.model_selection import StratifiedKFold

X = df['text_lemma']
y = df['label']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

CLASS_NAMES = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}

print('Distribuição por fold (Stratified K-Fold, k=5):')
print(f"{'Fold':<8} {'Train':>8} {'Val':>8} {'Bearish%':>10} {'Bullish%':>10} {'Neutral%':>10}")
print('-' * 60)

folds = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    y_val_fold = y.iloc[val_idx]
    pcts = y_val_fold.value_counts(normalize=True).sort_index() * 100
    folds.append((train_idx, val_idx))
    print(f"Fold {fold+1:<4} {len(train_idx):>8,} {len(val_idx):>8,} "
          f"{pcts.get(0,0):>9.1f}% {pcts.get(1,0):>9.1f}% {pcts.get(2,0):>9.1f}%")

print()
print('✅ Cada fold mantém a mesma proporção de classes (Stratified).')
print('   O balanceamento (class weights) será aplicado dentro de cada fold durante o treino.')

In [ ]:
# ── Guardar dataset processado ────────────────────────────────────────────────
df.to_csv('train_preprocessed.csv', index=False)
print(f'✅ Dataset preprocessado guardado: train_preprocessed.csv')
print(f'   Shape final: {df.shape}')
print(f'   Colunas: {list(df.columns)}')

usar as classes wheighs disse o claude que era o melhor 